In [ ]:
import io
import numpy as np
import zipfile
from pathlib import Path
import pandas as pd
import requests
from sqlalchemy import create_engine
import folium
import geopandas as gpd
from shapely.geometry import LineString, Point

In [ ]:
DB_DIR = Path.cwd() / "data"
DB_DIR.mkdir(exist_ok=True)
DB_PATH = DB_DIR / "ctrain.db"


In [ ]:
engine = create_engine(f"sqlite:///{DB_PATH}")

In [ ]:
GTFS_URL = "https://data.calgary.ca/download/npk7-z3bj/application%2Fzip"

response = requests.get(GTFS_URL)
if response.status_code != 200:
    raise Exception(
        f"Failed to fetch GTFS zip file. HTTP Status: {response.status_code}"
    )

zip_file = zipfile.ZipFile(io.BytesIO(response.content))

In [ ]:
def gtfs_time_to_seconds(time_str):
    if pd.isna(time_str):
        return None
    h, m, s = map(int, str(time_str).strip().split(":"))
    return h * 3600 + m * 60 + s


In [ ]:
routes_df = pd.read_csv(zip_file.open("routes.txt"), dtype=str)
ctrain_routes = routes_df[routes_df["route_short_name"].isin(["201", "202"])]
ctrain_route_ids = ctrain_routes["route_id"].unique()
ctrain_routes.to_sql("routes", engine, if_exists="replace", index=False)
ctrain_routes

In [ ]:
trips_df = pd.read_csv(zip_file.open("trips.txt"), dtype=str)
ctrain_trips = trips_df[trips_df["route_id"].isin(ctrain_route_ids)]
ctrain_trip_ids = ctrain_trips["trip_id"].unique()
ctrain_trips.to_sql("trips", engine, if_exists="replace", index=False)
ctrain_trips

In [ ]:
stop_times_df = pd.read_csv(zip_file.open("stop_times.txt"), dtype=str)
ctrain_stop_times = stop_times_df[
    stop_times_df["trip_id"].isin(ctrain_trip_ids)
].copy()
ctrain_stop_times["arrival_sec"] = ctrain_stop_times["arrival_time"].apply(
    gtfs_time_to_seconds
)
ctrain_stop_times["departure_sec"] = ctrain_stop_times["departure_time"].apply(
    gtfs_time_to_seconds
)
ctrain_stop_times.to_sql(
    "stop_times", engine, if_exists="replace", index=False
)
ctrain_stop_times

In [ ]:
stops_df = pd.read_csv(zip_file.open("stops.txt"), dtype=str)
ctrain_stops = stops_df[
    stops_df["stop_id"].isin(ctrain_stop_times["stop_id"].unique())
]
ctrain_stops.to_sql("stops", engine, if_exists="replace", index=False)
ctrain_stops

In [ ]:
shapes_df = pd.read_csv(zip_file.open("shapes.txt"), dtype=str)
ctrain_shapes = shapes_df[
    shapes_df["shape_id"].isin(ctrain_trips["shape_id"].dropna().unique())
]
ctrain_shapes.to_sql("shapes", engine, if_exists="replace", index=False)
ctrain_shapes

In [ ]:
# Check station names in the database
inspect_stops_query = """
SELECT DISTINCT stop_id, stop_name, stop_lat, stop_lon
FROM stops
ORDER BY stop_name;
"""

df_stops = pd.read_sql_query(inspect_stops_query, engine)
df_stops

In [ ]:
free_fare_query = """
WITH scheduled_arrivals AS (
    SELECT 
        s.stop_id,
        s.stop_name,
        r.route_short_name,
        st.arrival_sec,
        st.arrival_sec - LAG(st.arrival_sec) OVER (
            PARTITION BY st.stop_id, r.route_short_name 
            ORDER BY st.arrival_sec
        ) AS scheduled_headway_sec
    FROM stop_times st
    JOIN trips t ON st.trip_id = t.trip_id
    JOIN routes r ON t.route_id = r.route_id
    JOIN stops s ON st.stop_id = s.stop_id
    WHERE s.stop_name LIKE '%Free Fare Zone%' OR s.stop_name LIKE '%Free Fare%'
)
SELECT 
    stop_name,
    route_short_name,
    COUNT(*) AS total_scheduled_trips,
    ROUND(AVG(scheduled_headway_sec) / 60.0, 2) AS avg_headway_min,
    ROUND(MIN(scheduled_headway_sec) / 60.0, 2) AS min_headway_min,
    ROUND(MAX(scheduled_headway_sec) / 60.0, 2) AS max_headway_min
FROM scheduled_arrivals
WHERE scheduled_headway_sec IS NOT NULL
GROUP BY stop_name, route_short_name
ORDER BY stop_name, route_short_name;
"""

df_free_fare = pd.read_sql_query(free_fare_query, engine)
df_free_fare

In [ ]:
raw_headways_query = """
SELECT 
    s.stop_name,
    r.route_short_name,
    st.arrival_sec,
    (st.arrival_sec - LAG(st.arrival_sec) OVER (
        PARTITION BY st.stop_id, r.route_short_name 
        ORDER BY st.arrival_sec
    )) / 60.0 AS headway_min
FROM stop_times st
JOIN trips t ON st.trip_id = t.trip_id
JOIN routes r ON t.route_id = r.route_id
JOIN stops s ON st.stop_id = s.stop_id
WHERE s.stop_name LIKE '%Free Fare Zone%' OR s.stop_name LIKE '%Free Fare%'
"""

df_raw = pd.read_sql_query(raw_headways_query, engine).dropna()


def calculate_swt(headways):
    if len(headways) == 0 or sum(headways) == 0:
        return 0
    return sum(headways**2) / (2 * sum(headways))


results = []
for (stop, route), group in df_raw.groupby(["stop_name", "route_short_name"]):
    headways = group["headway_min"].values
    mean_h = np.mean(headways)
    swt = calculate_swt(headways)
    results.append(
        {
            "Stop Name": stop,
            "Route": route,
            "Mean Headway (min)": round(mean_h, 2),
            "Scheduled Wait Time (SWT min)": round(swt, 2),
            "Variance Penalty (min)": round(swt - (mean_h / 2), 2),
        }
    )

df_ewt_base = pd.DataFrame(results)
df_ewt_base

In [ ]:
shapes_df = pd.read_sql_query(
    "SELECT shape_id, shape_pt_lat, shape_pt_lon, shape_pt_sequence FROM shapes ORDER BY shape_id, CAST(shape_pt_sequence AS INTEGER)",
    engine,
)
stops_df = pd.read_sql_query(
    "SELECT stop_id, stop_name, stop_lat, stop_lon FROM stops", engine
)

stops_summary = pd.merge(stops_df, df_ewt_base, left_on="stop_name", right_on="Stop Name", how="left")
stops_summary

In [ ]:
lines = []
for shape_id, group in shapes_df.groupby("shape_id"):
    if len(group) > 1:
        points = [
            Point(float(lon), float(lat))
            for lat, lon in zip(group["shape_pt_lat"], group["shape_pt_lon"])
        ]
        lines.append({"shape_id": shape_id, "geometry": LineString(points)})

gdf_lines = gpd.GeoDataFrame(lines, crs="EPSG:4326")

In [ ]:
calgary_map = folium.Map(
    location=[51.046, -114.071], zoom_start=14, tiles="cartodbpositron"
)

In [ ]:
for _, row in gdf_lines.iterrows():
    sim_geo = gpd.GeoSeries(row["geometry"])
    geo_j = sim_geo.to_json()
    geo_j = folium.GeoJson(
        data=geo_j, style_function=lambda x: {"color": "#003366", "weight": 3, "opacity": 0.6}
    )
    geo_j.add_to(calgary_map)

In [ ]:
for _, row in stops_summary.iterrows():
    lat, lon = float(row["stop_lat"]), float(row["stop_lon"])
    is_free_fare = "Free Fare" in str(row["stop_name"])

    marker_color = "red" if is_free_fare else "blue"
    radius = 7 if is_free_fare else 4

    popup_text = f"""
    <b>{row['stop_name']}</b><br>
    <b>Route:</b> {row.get('Route', 'CTrain System')}<br>
    <b>Mean Headway:</b> {row.get('Mean Headway (min)', 'N/A')} min<br>
    <b>Scheduled Wait Time (SWT):</b> {row.get('Scheduled Wait Time (SWT min)', 'N/A')} min<br>
    <b>Variance Penalty:</b> {row.get('Variance Penalty (min)', 'N/A')} min
    """

    folium.CircleMarker(
        location=[lat, lon],
        radius=radius,
        popup=folium.Popup(popup_text, max_width=300),
        color=marker_color,
        fill=True,
        fill_color=marker_color,
        fill_opacity=0.8,
    ).add_to(calgary_map)

calgary_map.save("data/ctrain_free_fare_map.html")
calgary_map